Preparazione dati

1.Importo le librerie e controllo GPU

In [ ]:
import pandas as pd
import numpy as np
import re
import torch

print(f"PyTorch Version: {torch.__version__}")
#imposto la GPU come device principale (per usarlo sulla mia macchina, se non è disponibile userà la CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sto usando: {device}")

2. Analisi e preprocessing dei dati (caricamento del file berg.txt e pulizia del testo)

In [ ]:
def preprocess_sentence(w):
    #trasforma tutto in minuscolo e toglie spazi all'inizio e alla fine
    w = str(w).lower().strip()

    #1. sostituzione dei i trattini del discorso diretto con uno spazio
    w = re.sub(r'[–—―]', ' ', w)

    #2. separazione della punteggiatura dalle parole
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)

    #3. mantengo lettere, numeri, punteggiatura e il trattino corto
    #sono inclusi tutti gli accenti bergamaschi e italiani
    w = re.sub(r"[^a-zàèéìíòóùúöüåśź0-9?.!,¿'\-]+", " ", w)    
    w = w.strip()
    
    #inserimento dei token per far capire alla rete quando inizia e finisce la frase
    w = '<sos> ' + w + ' <eos>'
    return w

#test per vedere il risultato
test_phrase = "«L'è svèelt com ü fölmen.» – disse l'uomo" 
print("Prima:", test_phrase)
print("Dopo: ", preprocess_sentence(test_phrase))

3. Architettura del vocabolario (tokenizzazione e indicizzazione)


In [ ]:
from sklearn.model_selection import train_test_split

#leggo il file ignorando la terza colonna (fonti)
df = pd.read_csv("fiabe_berg.txt", sep="\t", header=None, usecols=[0, 1], names=["Bergamasco", "Italiano"])
df = df.dropna()

#preprocessing di entrambe le colonne
df['Bergamasco_pulito'] = df['Bergamasco'].apply(preprocess_sentence)
df['Italiano_pulito'] = df['Italiano'].apply(preprocess_sentence)

#spliting del dataset in train 80%, validation e test 20%
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)

#splitting ulteriore del 20% in validation e test finale con 10% ciascuno
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)

print(f"Totale frasi nel dataset: {len(df)}")
print(f"Frasi per Addestramento (Train): {len(df_train)}")
print(f"Frasi per Validazione (Val):   {len(df_val)}")
print(f"Frasi per Test Finale (Test):  {len(df_test)}")

In [ ]:
class Vocabulary:
    def __init__(self):
        #inizializzazione dei dizionari con i 4 token di base
        self.word2index = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.index2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.n_words = 4  #contatore delle parole (parte da 4 perché ci sono già i token)

    def add_sentence(self, sentence):
        #frase divisa in singole parole (sugli spazi)
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        #se la parola non è mai stata vista, viene inserita e le viene assegnato un ID
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

#si creano i due vocabolari vuoti per le due lingue
vocab_berg = Vocabulary()
vocab_ita = Vocabulary()

#popolo i vocabolari SOLO con i dati di Training
for frase in df_train['Bergamasco_pulito']:
    vocab_berg.add_sentence(frase)

for frase in df_train['Italiano_pulito']:
    vocab_ita.add_sentence(frase)

print(f"Grandezza vocabolario Bergamasco: {vocab_berg.n_words} parole uniche")
print(f"Grandezza vocabolario Italiano: {vocab_ita.n_words} parole uniche")

4. Salvo gli splitting

In [ ]:
import pickle

#salva i Dataframe puliti in 3 file CSV separati
df_train.to_csv("train_dataset.csv", index=False)
df_val.to_csv("val_dataset.csv", index=False)
df_test.to_csv("test_dataset.csv", index=False)

#salva i due oggetti Vocabulary usando la libreria 'pickle' di Python
with open('vocab_berg.pkl', 'wb') as f:
    pickle.dump(vocab_berg, f)

with open('vocab_ita.pkl', 'wb') as f:
    pickle.dump(vocab_ita, f)

print("Tutto salvato con successo!")
print("File creati: train_dataset.csv, val_dataset.csv, test_dataset.csv, vocab_berg.pkl, vocab_ita.pkl")